# Regional compound-event ECA (sliding 30-year windows)

Reproduces the paper's Fig. 4 (per-region band + observed coincidences, per-ensemble
significance heatmaps) and Fig. 5 (regional ensemble-agreement maps) using the vendored
engine in `eca_analysis` (see `eca_analysis/VENDORED.md`).

Method (paper Sections 2.1-2.3):
- extreme day = precipitation > threshold (fixed 20 mm/hr, or the thermodynamically
  adjusted per-window value matching the baseline percentile)
- self-ECA with `delT=4, tau=1`; **observed coincidences never bridge a season
  boundary** (year-blocked)
- null band = 2.5-97.5% of Eq. (1)'s binomial (`null_model="pooled"`), K indexed from 0

In [ ]:
# imports & config
import sys
sys.path.insert(0, "..")

import yaml
import pandas as pd

from eca_analysis import WindowConfig, run_window_analysis, plot_region_4panel

with open("../config/config.yaml") as f:
    cfg_yaml = yaml.safe_load(f)

BASE_DIR     = cfg_yaml["data"]["base_dir"]
GEOJSON_PATH = cfg_yaml["data"]["geojson_path"]
SUMMARY_CSV  = cfg_yaml["output"]["summary_csv"]
FIGURES_DIR  = cfg_yaml["output"]["figures_dir"]
A = cfg_yaml["analysis"]

In [ ]:
# regions and ensemble members
regions = {
    'East_Midlands': ['East Midlands'], 'East_Scotland': ['East Scotland'],
    'East_of_England': ['East of England'], 'North_East_England': ['North East England'],
    'North_Scotland': ['North Scotland'], 'North_West_England': ['North West England'],
    'Northern_Ireland': ['Northern Ireland'], 'South_East_England': ['South East England'],
    'South_West_England': ['South West England'], 'Wales': ['Wales'],
    'West_Midlands': ['West Midlands'], 'West_Scotland': ['West Scotland'],
    'Yorkshire_and_Humber': ['Yorkshire and Humber'],
}
ensembles = ['0000', '1113', '1554', '1649', '1843', '1935',
             '2123', '2242', '2305', '2335', '2491', '2868']

cfg = WindowConfig(
    data_dir=BASE_DIR,
    out_dir=FIGURES_DIR,
    ensembles=ensembles,
    regions=list(regions.keys()),
    region_names={k: v[0] for k, v in regions.items()},
    fixed_threshold=A["wet_threshold"],
    delT=A["delT"],
    tau=A["tau"],
    len_wet=A["len_wet"],
    year_blocked=True,               # paper: no Sep -> next-June coincidences
    null_model=A.get("null_model", "pooled"),   # "pooled" = paper Eq. (1)
)

In [ ]:
# run: every ensemble x region x 30-yr window, fixed + thermodynamic regimes.
# Writes per-ensemble multi-panel figures and a tidy CSV; returns the tidy frame.
windows_df = run_window_analysis(cfg)
windows_df.head()

In [ ]:
# single-region 4-panel (paper Fig. 4): band + points for one member,
# significance heatmaps across all members
plot_region_4panel(windows_df, cfg, region_code="Wales", ensemble="0000",
                   out_path=f"{FIGURES_DIR}/wales_4panel.png",
                   region_display="Wales")

In [ ]:
# regional ensemble-agreement maps (paper Fig. 5)
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt

from EventCoincidenceAnalysis.functions.plotting import (
    ensemble_agreement_summary, plot_regional_ensemble_agreement)

summary_df = ensemble_agreement_summary(
    windows_df, region_names={k: v[0] for k, v in regions.items()})

Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Saved summary to {SUMMARY_CSV}")
print(summary_df)

regions_gdf = gpd.read_file(GEOJSON_PATH)
fig = plot_regional_ensemble_agreement(summary_df, regions_gdf)
fig.savefig(f"{FIGURES_DIR}/regional_ensemble_agreement.png",
            dpi=150, bbox_inches="tight")
plt.show()